<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/report_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CLONE GITHUB REPO**

In [1]:
import os
import sys
from google.colab import userdata

# 1. Configuration
USERNAME = "Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"
ROOT_PATH = f'/content/{REPO_NAME}'

try:
    token = userdata.get('GITHUB_TOKEN')
    AUTH_REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

    # 2. Clean/Sync Repository
    if not os.path.exists(ROOT_PATH):
        !git clone {AUTH_REPO_URL}
    else:
        %cd {ROOT_PATH}
        !git remote set-url origin {AUTH_REPO_URL}
        !git fetch origin
        !git reset --hard origin/main

    # 3. Path Initialization
    if ROOT_PATH not in sys.path:
        sys.path.append(ROOT_PATH)

    MACH3_ROOT = os.path.join(ROOT_PATH, 'Power BI/Automations/Power Bi Desktop/Mach3')
    %cd "{MACH3_ROOT}"

    print(f"Environment Reinitialized.\nRoot: {ROOT_PATH}\nWorking Dir: {os.getcwd()}")
except Exception as e:
    print(f"Initialization Error: {e}")

Cloning into 'Power_BI_Spark_Labs'...
remote: Enumerating objects: 1838, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 1838 (delta 63), reused 47 (delta 24), pack-reused 1716 (from 1)
Receiving objects: 100% (1838/1838), 28.28 MiB | 27.35 MiB/s, done.
Resolving deltas: 100% (843/843), done.
/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3
Environment Reinitialized.
Root: /content/Power_BI_Spark_Labs
Working Dir: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


# **FUNCTIONS DECLARATION**

In [42]:
import zipfile
import shutil
import os
import json
import pandas as pd
from mach3_core import FabricReport
from fabric_models import Report, Page, VisualContainer, Bookmark

def check_models_gen():
  model_path = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/fabric_models.py'
  with open(model_path, 'r') as f:
      first_lines = [next(f) for _ in range(1)]
  print(''.join(first_lines))
  return

def delete(folder_path):
  if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
  return f"{folder_path} deleted"

def zip(folder_path, zip_path):
    shutil.make_archive(zip_path, 'zip', folder_path)
    return zip_path + '.zip' + 'created'

def unzip(zip_path = '/content/definition.zip', extract_path = '/content/definition'):
  if os.path.exists(zip_path):
      with zipfile.ZipFile(zip_path, 'r') as zip_ref:
          zip_ref.extractall(extract_path)
      print(f"Extracted {zip_path} to {extract_path}/")
  else:
      print(f"Zip file not found at {zip_path}.")

class FabricBPARules:
    def __init__(self, report_obj, static_files=None):
        self.report = report_obj
        self.static_files = static_files or {}
        self.short_types = {
            'visual': 'Vis', 'slicer': 'Slcr', 'textbox': 'Txt', 'actionButton': 'Btn',
            'image': 'Img', 'shape': 'Shp', 'card': 'Crd', 'lineChart': 'Line',
            'barChart': 'Bar', 'pieChart': 'Pie', 'table': 'Tbl', 'pivotTable': 'Mtx'
        }

    @staticmethod
    def load_from_folder(src_path):
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"Folder not found: {src_path}")
        pages_list = []
        bookmarks_list = []
        static_files = {}

        for f_name in ['version.json']:
            f_path = os.path.join(src_path, f_name)
            if os.path.exists(f_path):
                with open(f_path, 'r') as f: static_files[f_name] = json.load(f)

        with open(os.path.join(src_path, 'report.json'), 'r') as f:
            master_report = Report(**json.load(f))

        pages_dir = os.path.join(src_path, 'pages')
        if os.path.exists(pages_dir):
            for p_folder in os.listdir(pages_dir):
                folder_path = os.path.join(pages_dir, p_folder)
                if not os.path.isdir(folder_path): continue
                with open(os.path.join(folder_path, 'page.json'), 'r') as f:
                    page_obj = Page(**json.load(f))
                v_list = []
                v_dir = os.path.join(folder_path, 'visuals')
                if os.path.exists(v_dir):
                    for v_f in os.listdir(v_dir):
                        v_path = os.path.join(v_dir, v_f, 'visual.json')
                        if os.path.exists(v_path):
                            with open(v_path, 'r') as f:
                                v_list.append(VisualContainer(**json.load(f)))
                pages_list.append((page_obj, v_list))

        bookmarks_dir = os.path.join(src_path, 'bookmarks')
        if os.path.exists(bookmarks_dir):
            bm_meta = os.path.join(bookmarks_dir, 'bookmarks.json')
            if os.path.exists(bm_meta):
                with open(bm_meta, 'r') as f: static_files['bookmarks/bookmarks.json'] = json.load(f)
            for b_file in os.listdir(bookmarks_dir):
                if b_file.endswith('.bookmark.json'):
                    with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
                        bookmarks_list.append(Bookmark(**json.load(f)))

        report_obj = FabricReport(master_report, pages_list, bookmarks_list)
        return FabricBPARules(report_obj, static_files)

    def _get_visual_metadata(self, visual):
        v_data = visual.root.visual
        if v_data is None: return "unknown", ""
        v_type = getattr(v_data, 'visualType', getattr(v_data, 'visual_type', 'unknown'))
        v_display = ""
        try:
            v_dict = v_data.model_dump(by_alias=True) if hasattr(v_data, 'model_dump') else v_data
            title_obj = v_dict.get('visualContainerObjects', {}).get('title', [])[0]
            v_display = title_obj.get('properties', {}).get('text', {}).get('expr', {}).get('Literal', {}).get('Value', '').strip("'")
        except: v_display = ""
        return v_type, v_display

    def _sync_id_references(self, rename_map):
        """Deep textual replacement across all model attributes."""
        if not rename_map: return
        # Process bookmarks
        for bm in self.report.bookmarks:
            s = json.dumps(bm.model_dump(by_alias=True, exclude_none=True, mode='json'))
            for o, n in rename_map.items(): s = s.replace(f'"{o}"', f'"{n}"')
            bm.__dict__.update(Bookmark(**json.loads(s)).__dict__)
        # Process pages and visuals
        for p_wrapper in self.report.pages:
            p_str = json.dumps(p_wrapper.model.model_dump(by_alias=True, exclude_none=True, mode='json'))
            for o, n in rename_map.items(): p_str = p_str.replace(f'"{o}"', f'"{n}"')
            p_wrapper.model = Page(**json.loads(p_str))
            for v in p_wrapper.visuals:
                v_str = json.dumps(v.model_dump(by_alias=True, exclude_none=True, mode='json'))
                for o, n in rename_map.items(): v_str = v_str.replace(f'"{o}"', f'"{n}"')
                v.__dict__.update(VisualContainer(**json.loads(v_str)).__dict__)

    def fix_page_names(self, src_path):
        pages_json_path = os.path.join(src_path, 'pages', 'pages.json')
        if not os.path.exists(pages_json_path): ordered_ids = [p.model.name for p in self.report.pages]
        else:
            with open(pages_json_path, 'r') as f: ordered_ids = json.load(f).get('pageOrder', [])
        self.report.pages.sort(key=lambda p: ordered_ids.index(p.model.name) if p.model.name in ordered_ids else 999)
        rename_map = {}
        for i, p in enumerate(self.report.pages, 1):
            old, new = p.model.name, f"P{i}"
            if old != new: rename_map[old] = new; p.model.name = new
        if rename_map: self._sync_id_references(rename_map)
        return rename_map

    def fix_visual_names(self):
        rename_map = {}
        for page_wrapper in self.report.pages:
            page_display = "".join(e for e in page_wrapper.model.displayName if e.isalnum())[:10]
            counter = 1
            for visual in page_wrapper.visuals:
                old_name = visual.root.name
                v_type, v_display = self._get_visual_metadata(visual)
                v_short = self.short_types.get(v_type, v_type[:4])
                clean_display = "".join(e for e in v_display if e.isalnum())[:15]
                new_name = f"{page_display}_{v_short}_{clean_display}_{counter}"[:49]
                if old_name != new_name:
                    visual.root.name = new_name
                    rename_map[old_name] = new_name
                counter += 1
        self._sync_id_references(rename_map)
        return rename_map

    def fix_bookmark_names(self):
        rename_map = {}
        for i, bm in enumerate(self.report.bookmarks, 1):
            old, new = bm.name, f"Bmk{i}"
            if old != new: rename_map[old] = new; bm.name = new
        self._sync_id_references(rename_map)
        return rename_map

    def save_report(self, output_path):
        if os.path.exists(output_path): shutil.rmtree(output_path)
        os.makedirs(output_path)
        for rel_path, content in self.static_files.items():
            full_path = os.path.join(output_path, rel_path)
            os.makedirs(os.path.dirname(full_path), exist_ok=True)
            with open(full_path, 'w') as f: json.dump(content, f, indent=2)
        master = getattr(self.report, 'metadata', getattr(self.report, 'report', None))
        if master:
            with open(os.path.join(output_path, 'report.json'), 'w') as f:
                json.dump(master.model_dump(by_alias=True, exclude_none=True, mode='json'), f, indent=2)
        pages_dir = os.path.join(output_path, 'pages')
        os.makedirs(pages_dir, exist_ok=True)
        p_idx = {
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json",
            "pageOrder": [p.model.name for p in self.report.pages],
            "activePageName": self.report.pages[0].model.name if self.report.pages else None
        }
        with open(os.path.join(pages_dir, 'pages.json'), 'w') as f: json.dump(p_idx, f, indent=2)
        for pw in self.report.pages:
            p_path = os.path.join(pages_dir, pw.model.name)
            os.makedirs(p_path)
            with open(os.path.join(p_path, 'page.json'), 'w') as f:
                json.dump(pw.model.model_dump(by_alias=True, exclude_none=True, mode='json'), f, indent=2)
            if pw.visuals:
                v_dir = os.path.join(p_path, 'visuals')
                os.makedirs(v_dir)
                for v in pw.visuals:
                    vp = os.path.join(v_dir, v.root.name)
                    os.makedirs(vp)
                    with open(os.path.join(vp, 'visual.json'), 'w') as f:
                        json.dump(v.model_dump(by_alias=True, exclude_none=True, mode='json'), f, indent=2)
        if self.report.bookmarks:
            b_dir = os.path.join(output_path, 'bookmarks')
            os.makedirs(b_dir, exist_ok=True)
            for bm in self.report.bookmarks:
                with open(os.path.join(b_dir, f'{bm.name}.bookmark.json'), 'w') as f:
                    json.dump(bm.model_dump(by_alias=True, exclude_none=True, mode='json'), f, indent=2)

### **Integrated Report Builder Classes**
This section adapts the provided Top-Down generation logic to use the `fabric_models` Pydantic classes.

In [83]:
import copy
from fabric_models import Report as ReportModel, Page as PageModel, VisualContainer, VisualContainer1, VisualContainer2

class LayoutEngine:
    @staticmethod
    def get_v1_coordinates(canvas_w=1280, canvas_h=720, padding=2, chapters_cnt=5):
        # Logic from your provided generate_layout_v1
        hdr_h = 0.15 * canvas_h
        hdr_main_h = 0.50 * hdr_h
        left_w = 0.10 * canvas_w

        coords = {
            "org_logo": {"x": padding, "y": padding, "w": left_w - 2*padding, "h": hdr_main_h - 2*padding, "z": 0},
            "report_title": {"x": left_w + padding, "y": padding, "w": 0.25 * (canvas_w - left_w), "h": hdr_main_h - 2*padding, "z": 100},
        }
        return coords

class FabricVisual:
    """Base class that wraps a Pydantic VisualContainer"""
    def __init__(self, name, v_type, coords=None):
        self.name = name
        self.v_type = v_type
        # Initialize a default Pydantic model for the visual with the required $schema
        self.model = VisualContainer(root=VisualContainer1(**{
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/1.0.0/schema.json",
            "name": name,
            "position": {"x": 0, "y": 0, "z": 0, "width": 100, "height": 100, "tabOrder": 0},
            "visual": {"visualType": v_type, "objects": {}, "visualContainerObjects": {}}
        }))
        if coords:
            self.set_position(coords.get('x'), coords.get('y'), coords.get('w'), coords.get('h'), coords.get('z'))

    def set_position(self, x, y, w, h, z=0):
        # Handle both object-attribute access and dictionary access for robustness
        pos = self.model.root.position
        if isinstance(pos, dict):
            pos['x'], pos['y'], pos['width'], pos['height'], pos['z'] = x, y, w, h, z
        else:
            pos.x, pos.y, pos.width, pos.height, pos.z = x, y, w, h, z

class FabricTextBox(FabricVisual):
    def __init__(self, name, text, coords=None):
        super().__init__(name, "textbox", coords)
        self.text = text
        self.update_text()

    def update_text(self):
        self.model.root.visual['objects']['general'] = [{
            "properties": {"paragraphs": [{"textRuns": [{"value": self.text}]}]}
        }]

class FabricPage:
    def __init__(self, name, display_name=None):
        # Fix: PageModel requires $schema and displayOption
        self.model = PageModel(**{
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/1.0.0/schema.json",
            "name": name,
            "displayName": display_name or name,
            "displayOption": "fitToPage"
        })
        self.visuals = []

    def add_visual(self, visual_obj):
        self.visuals.append(visual_obj)

class FabricBuilderReport:
    def __init__(self, name):
        # Use the alias '$schema' as required by the generated Pydantic models
        self.report_model = ReportModel(**{
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/report/1.0.0/schema.json",
            "themeCollection": {}
        })
        self.pages = []

    def add_page(self, page_obj):
        self.pages.append(page_obj)

    def to_fabric_report(self):
        pages_list = []
        for p in self.pages:
            v_models = [v.model for v in p.visuals]
            pages_list.append((p.model, v_models))
        return FabricReport(self.report_model, pages_list, [])

In [84]:
# 1. Initialize the Layout Engine and Report Builder
layout = LayoutEngine()
builder = FabricBuilderReport(name="Automated Sample Report")

# 2. Define coordinates for visuals
canvas_coords = layout.get_v1_coordinates()

# 3. Create a Page and add visuals to it
sample_page = FabricPage(name="P1", display_name="Executive Summary")

# Add a Text Box visual using coordinates from the layout engine
title_visual = FabricTextBox(
    name="report_title_v1",
    text="Sales Performance Dashboard",
    coords=canvas_coords.get("report_title")
)
sample_page.add_visual(title_visual)

# 4. Add the page to the report
builder.add_page(sample_page)

# 5. Convert to the core FabricReport object used by BPA rules
fabric_report = builder.to_fabric_report()

# 6. Save the report to a definition folder
output_path = '/content/builder_sample_output'
# We use the BPA rules class to handle the directory structure and schema serialization
engine = FabricBPARules(fabric_report)
engine.save_report(output_path)

print(f"Sample report generated and saved to: {output_path}")
fabric_report.get_summary()

Sample report generated and saved to: /content/builder_sample_output
--- Fabric Report Master Summary ---
Pages: 1 | Bookmarks: 0
- Executive Summary (1 visuals)


### **1. Advanced Layout Definition**
We can extend the `LayoutEngine` to create specific templates (e.g., Executive, Operational). This keeps the visual coordinates separate from the report logic.

In [85]:
class ExecutiveLayout(LayoutEngine):
    @staticmethod
    def get_dashboard_coords(canvas_w=1280, canvas_h=720):
        # Define dynamic regions
        padding = 10
        banner_h = 60
        kpi_w = (canvas_w - (4 * padding)) / 3

        return {
            "header": {"x": 0, "y": 0, "w": canvas_w, "h": banner_h, "z": 0},
            "kpi_1": {"x": padding, "y": banner_h + padding, "w": kpi_w, "h": 120, "z": 10},
            "kpi_2": {"x": 2*padding + kpi_w, "y": banner_h + padding, "w": kpi_w, "h": 120, "z": 10},
            "main_chart": {"x": padding, "y": banner_h + 140, "w": canvas_w - 2*padding, "h": 400, "z": 5}
        }

### **2. Step-by-Step Report Construction**
This follows the flow: **Initialize Builder -> Get Layout -> Create Page -> Add Visuals -> Export.**

In [86]:
# Step 1: Initialize
builder = FabricBuilderReport(name="Executive Dashboard v2")
coords = ExecutiveLayout.get_dashboard_coords()

# Step 2: Create Page
exec_page = FabricPage(name="ExecSummary", display_name="Sales Overview")

# Step 3: Add Header Visual
exec_page.add_visual(FabricTextBox(
    name="header_txt",
    text="FY24 Q4 Performance",
    coords=coords['header']
))

# Step 4: Add KPI Placeholders
for i in range(1, 3):
    exec_page.add_visual(FabricTextBox(
        name=f"kpi_box_{i}",
        text=f"KPI {i}: $0.00M",
        coords=coords[f'kpi_{i}']
    ))

# Step 5: Finalize and Save
builder.add_page(exec_page)
final_report = builder.to_fabric_report()

output_dir = '/content/executive_layout_demo'
FabricBPARules(final_report).save_report(output_dir)

print(f"Report assembled using ExecutiveLayout at: {output_dir}")
final_report.get_summary()

Report assembled using ExecutiveLayout at: /content/executive_layout_demo
--- Fabric Report Master Summary ---
Pages: 1 | Bookmarks: 0
- Sales Overview (3 visuals)


### **3. Expanding Visual Type Support**
To handle more complex reports, we need specialized classes for different visual types that map to the Pydantic schema.

In [87]:
class FabricChart(FabricVisual):
    def __init__(self, name, chart_type, coords=None):
        super().__init__(name, chart_type, coords)

class FabricSlicer(FabricVisual):
    def __init__(self, name, coords=None):
        super().__init__(name, "slicer", coords)

# Example usage for a Bar Chart
# bar_chart = FabricChart("sales_bar", "barChart", coords=coords['main_chart'])

### **4. Synchronize to GitHub**
Now that we have verified the Top-Down generation and schema fixes, we push the updated logic and models to the repository.

In [88]:
from mach3_helpers import push_to_github

# Commit message reflecting the major structural upgrades
msg = "Enhance PBIR Builder: Top-Down Layouts, Navigator Sync, and Schema Fixes"
push_to_github(ROOT_PATH, commit_message=msg)

Push sequence complete.


In [ ]:
# def validate_report_schema(fabric_report_obj):
#     print("--- Starting Schema Validation ---")
#     try:
#         if hasattr(fabric_report_obj, 'metadata') and fabric_report_obj.metadata:
#             dump = fabric_report_obj.metadata.model_dump(by_alias=True)
#             fabric_report_obj.metadata.model_validate(dump)
#             print(f"✓ Metadata Schema: Valid")
#         for page_wrapper in fabric_report_obj.pages:
#             page_wrapper.model.model_validate(page_wrapper.model.model_dump(by_alias=True))
#             for v in page_wrapper.visuals:
#                 v.model_validate(v.model_dump(by_alias=True))
#         print(f"✓ {len(fabric_report_obj.pages)} Pages and associated Visuals: Valid")
#         for bookmark in fabric_report_obj.bookmarks:
#             bookmark.model_validate(bookmark.model_dump(by_alias=True))
#         print(f"✓ {len(fabric_report_obj.bookmarks)} Bookmarks: Valid")
#         print("\nSUCCESS: All components adhere to the fabric_models schema.")
#         return True
#     except Exception as e:
#         print(f"\nSCHEMA VALIDATION FAILED:")
#         print(str(e))
#         return False

# **GENERATE MODELS**

In [ ]:
%run power_bi_objects_creation.ipynb

# **IMPORT OBJECTS**

In [3]:
from fabric_models import Report, Page, VisualContainer, Bookmark
from mach3_core import  FabricReport,FabricBPARules
import os
import json


In [4]:
check_models_gen()

# Generated on: 2026-04-30 14:34:56 IST



In [13]:
import inspect
from fabric_models import Report

# Inspect the Report model fields to see defaults
report_fields = Report.model_fields
for field_name in ['report_source', 'organization_custom_visuals', 'annotations', 'data_source_variables']:
    if field_name in report_fields:
        field = report_fields[field_name]
        print(f"Field: {field_name}")
        print(f"  - Alias: {field.alias}")
        print(f"  - Default: {field.default}")
        print(f"  - Is Required: {field.is_required()}")
        print("-" * 20)

Field: annotations
  - Alias: None
  - Default: None
  - Is Required: False
--------------------


In [57]:
delete('/content/custom_layout_output')

'/content/custom_layout_output deleted'

## **Zip and UnZip Folders**

In [ ]:
zip( folder_path= '/content/definition', zip_path= '/content')

In [58]:
unzip(zip_path = '/content/base definition.zip',extract_path='/content/')

Extracted /content/base definition.zip to /content//


# **REPORT GEN AND TESTING**

## **Parse/ Load the Report from definition Folder**

In [ ]:
# src_path = os.path.join(MACH3_ROOT, 'definition')
# pages_list = []
# bookmarks_list = []

# if os.path.exists(src_path):
#     print(f"Loading definition from: {src_path}")
#     try:
#         with open(os.path.join(src_path, 'report.json'), 'r') as f:
#             master_report = Report(**json.load(f))

#         pages_dir = os.path.join(src_path, 'pages')
#         if os.path.exists(pages_dir):
#             for p_folder in os.listdir(pages_dir):
#                 folder_path = os.path.join(pages_dir, p_folder)
#                 if not os.path.isdir(folder_path): continue
#                 page_json_path = os.path.join(folder_path, 'page.json')
#                 if os.path.exists(page_json_path):
#                     with open(page_json_path, 'r') as f:
#                         page_obj = Page(**json.load(f))
#                     v_list = []
#                     v_dir = os.path.join(folder_path, 'visuals')
#                     if os.path.exists(v_dir):
#                         for v_f in os.listdir(v_dir):
#                             v_path = os.path.join(v_dir, v_f, 'visual.json')
#                             if os.path.exists(v_path):
#                                 with open(v_path, 'r') as f:
#                                     v_list.append(VisualContainer(**json.load(f)))
#                     pages_list.append((page_obj, v_list))

#         bookmarks_dir = os.path.join(src_path, 'bookmarks')
#         if os.path.exists(bookmarks_dir):
#             for b_file in os.listdir(bookmarks_dir):
#                 if b_file.endswith('.bookmark.json'):
#                     with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
#                         bookmarks_list.append(Bookmark(**json.load(f)))

#         report = FabricReport(master_report, pages_list, bookmarks_list)
#         print("\nSUCCESS: Report objects created and validated.")
#         report.get_summary()
#     except Exception as e:
#         print(f"Validation error details:\n{e}")
# else:
#     print(f"Definition folder not found at {src_path}.")

Loading definition from: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition

SUCCESS: Report objects created and validated.
--- Fabric Report Master Summary ---
Pages: 11 | Bookmarks: 26
- Transaction Trend (29 visuals)
- Private Label (28 visuals)
- Online (32 visuals)
- Insurance (34 visuals)
- Region (25 visuals)
- Units (36 visuals)
- Promo (37 visuals)
- S&OP Achievement (41 visuals)
- Sales (44 visuals)
- Enablers (49 visuals)
- Time-Based Analysis (24 visuals)


# **BPA RULES ENGINE DEFINITION**

### **How to use FabricBPARules**

To initialize and run the rules engine, follow these steps:

1. **Initialize**: Pass your `report` object to `FabricBPARules`.
2. **Validate**: Call `.validate_schema()` to ensure the report structure is correct.
3. **Analyze**: Call `.run_all_checks()` to find naming convention issues.
4. **Fix & Save**: Use `.fix_visual_names()` to rename and `.save_report()` to export the results.

In [29]:
# 1. Reload the report fresh with the updated static file handling
report_folder = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition'
engine = FabricBPARules.load_from_folder(report_folder)

# 2. Apply transformations
engine.fix_page_names(report_folder)
engine.fix_visual_names()
engine.fix_bookmark_names()

# 3. Save to a new verified path
verified_output_path = '/content/final_verified_definition_v8'
engine.save_report(verified_output_path)

# 4. Check for missing files again
print("--- Final Missing File Check ---")
find_missing_files(report_folder, verified_output_path)

--- Final Missing File Check ---
Source files count: 420
Output files count: 420

Potentially missing file types: {'95a77a16420ba7ad7e03.bookmark.json', 'defd0b45eaa160dd2628.bookmark.json', '1fe6d9ed1ce4e64e7b4a.bookmark.json', '793e27414cd27eb46d6a.bookmark.json', '5214773fe68b065a0dda.bookmark.json', '4cc0314d880280663807.bookmark.json', 'd812e04129c358c9e9ae.bookmark.json', '6c1a242a31167863b1e4.bookmark.json', '60e9829b038cd7c1a76c.bookmark.json', 'a55f32ac0c0e71b89000.bookmark.json', '3d7b4165a4532ea34a1b.bookmark.json', '56fbf99d4ce7862a0705.bookmark.json', 'e01eecda4034bbe2104c.bookmark.json', '9a64abf8a9dde13d4057.bookmark.json', 'a5330f6f0a233016cd79.bookmark.json', 'c57f7fe4dac90490ca42.bookmark.json', '41a4dd8390d013c2b289.bookmark.json', '3081b50dab41cd689d86.bookmark.json', 'de3a2c96c7b4d1097b4a.bookmark.json', 'af8f519bdbbd51b4a03a.bookmark.json', '231d6be9a9ce8129e9e1.bookmark.json', '595ae7f2921bc51cb4a6.bookmark.json', '7313d890147207137805.bookmark.json', '776a215db0

In [31]:
# Zipping the final processed output for download
zip_result = zip(folder_path='/content/final_verified_definition_v8', zip_path='/content/fixed_definition_v8')
print(zip_result)

/content/fixed_definition_v8.zipcreated


In [59]:
# 1. Initialize fresh report
src_folder = '/content/definition'
engine = FabricBPARules.load_from_folder(src_folder)
report = engine.report

# 2. Define your layout logic here
# Example: target_pages = ['Sales', 'Finance', 'Executive Summary']
# report.pages = [p for p in report.pages if p.model.displayName in target_pages]

# 3. Save the result
custom_layout_path = '/content/custom_layout_output'
engine.save_report(custom_layout_path)
print(f"Custom layout saved to: {custom_layout_path}")

Custom layout saved to: /content/custom_layout_output


In [60]:
report.get_summary()

--- Fabric Report Master Summary ---
Pages: 2 | Bookmarks: 0
- Homepage (7 visuals)
- Page (33 visuals)


In [62]:
import copy

# 1. Identify the reference page (the one named 'Page')
# Since report.pages contains wrappers, we access the .model and .visuals attributes
ref_wrapper = next(p for p in report.pages if p.model.displayName == 'Page')

# 2. Create a deep copy of the page model and its visuals
new_page_model = copy.deepcopy(ref_wrapper.model)
new_visuals = copy.deepcopy(ref_wrapper.visuals)

# 3. Update properties for the new page to avoid ID conflicts
new_page_model.name = f"P{len(report.pages) + 1}"
new_page_model.displayName = 'New Reference Page'

# 4. Use the type of the existing wrapper to ensure compatibility
PageWrapperClass = type(ref_wrapper)
report.pages.append(PageWrapperClass(new_page_model, new_visuals))

# 5. Save the updated report to the output folder
output_dir = '/content/definition_output'
engine.save_report(output_dir)

print(f"New page added and saved to {output_dir}")
print(f"Total pages now: {len(report.pages)}")
report.get_summary()

New page added and saved to /content/definition_output
Total pages now: 3
--- Fabric Report Master Summary ---
Pages: 3 | Bookmarks: 0
- Homepage (7 visuals)
- Page (33 visuals)
- New Reference Page (33 visuals)


In [63]:
import json
import os

pages_json_path = '/content/definition_output/pages/pages.json'

if os.path.exists(pages_json_path):
    with open(pages_json_path, 'r') as f:
        pages_meta = json.load(f)
    print("--- Contents of pages.json ---")
    print(json.dumps(pages_meta, indent=2))
else:
    print(f"File not found: {pages_json_path}")

--- Contents of pages.json ---
{
  "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json",
  "pageOrder": [
    "222b24aa54a5c20a9775",
    "ddfd06f072766000d7d7",
    "P3"
  ],
  "activePageName": "222b24aa54a5c20a9775"
}


In [67]:
# 1. Verify visuals existence
target_page_wrapper = next(p for p in report.pages if p.model.name == 'P3')
print(f"Total visuals on P3: {len(target_page_wrapper.visuals)}")

if len(target_page_wrapper.visuals) > 0:
    # 2. Inspect the first VisualContainer's entire structure
    first_v = target_page_wrapper.visuals[0]
    print("\nRaw VisualContainer dump:")
    # Using model_dump on the container itself to see the hierarchy
    full_dump = first_v.model_dump(by_alias=True) if hasattr(first_v, 'model_dump') else str(first_v)
    import json
    print(json.dumps(full_dump, indent=2)[:1000])

    # 3. Check for the 'visual' property specifically in the root
    if hasattr(first_v, 'root') and first_v.root:
        print("\nRoot attributes:", dir(first_v.root))
        if hasattr(first_v.root, 'visual'):
             print("Visual attribute content type:", type(first_v.root.visual))
else:
    print("ERROR: No visuals found on the new page. The copy might have failed.")

Total visuals on P3: 33

Raw VisualContainer dump:
{
  "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.6.0/schema.json",
  "name": "c4b761353979b047d6e0",
  "position": {
    "x": 341.69418416305575,
    "y": 71.13315061429297,
    "z": 5000,
    "height": 17.292352269221595,
    "width": 113.81411741232971,
    "tabOrder": 26
  },
  "visual": {
    "visualType": "actionButton",
    "objects": {
      "icon": [
        {
          "properties": {
            "shapeType": {
              "expr": {
                "Literal": {
                  "Value": "'blank'"
                }
              }
            }
          },
          "selector": {
            "id": "default"
          }
        },
        {
          "properties": {
            "show": {
              "expr": {
                "Literal": {
                  "Value": "false"
                }
              }
            }
          }
        }
      ],
      "outline

In [69]:
# Find the Page Navigator and adjust its navigation target
target_page_wrapper = next(p for p in report.pages if p.model.name == 'P3')

def find_and_update_navigator(visual_container):
    # Access the visual content safely
    if not hasattr(visual_container, 'root') or not visual_container.root:
        return False

    # The parser might store visuals in different fields based on the schema version
    v_obj = getattr(visual_container.root, 'visual', None)
    if v_obj is None:
        return False

    # Convert to dict for flexible property checking
    v_dict = v_obj if isinstance(v_obj, dict) else v_obj.model_dump(by_alias=True)

    v_type = v_dict.get('visualType') or v_dict.get('visual_type')
    if v_type == 'pageNavigator':
        print(f"Found Navigator in visual: {visual_container.root.name}")
        # Print objects to see navigation settings
        print("Navigator Configuration:")
        import json
        print(json.dumps(v_dict.get('objects', {}), indent=2))
        return True
    return False

print(f"Scanning {len(target_page_wrapper.visuals)} visuals...")
found_any = False
for v in target_page_wrapper.visuals:
    if find_and_update_navigator(v):
        found_any = True
        break

if not found_any:
    print("No visual with type 'pageNavigator' found. Checking for button-based navigation.")

Scanning 33 visuals...
Found Navigator in visual: 9b6b58405bb1540402d5
Navigator Configuration:
{
  "shape": [
    {
      "properties": {
        "tileShape": {
          "expr": {
            "Literal": {
              "Value": "'tabRoundTopCorners'"
            }
          }
        }
      },
      "selector": {
        "id": "default"
      }
    }
  ],
  "pages": [
    {
      "properties": {
        "showPage": {
          "expr": {
            "Literal": {
              "Value": "false"
            }
          }
        }
      },
      "selector": {
        "id": "48af020e0c960347411d"
      }
    },
    {
      "properties": {
        "showPage": {
          "expr": {
            "Literal": {
              "Value": "false"
            }
          }
        }
      },
      "selector": {
        "id": "e75757662cd82ca3f763"
      }
    },
    {
      "properties": {
        "showPage": {
          "expr": {
            "Literal": {
              "Value": "false"
            }


In [70]:
# Function to enable visibility for the new page in the Navigator
def enable_page_in_navigator(visual_container, target_page_id):
    # 1. Access the visual model
    v_obj = visual_container.root.visual

    # 2. Extract objects - handle both dict and Pydantic models
    if hasattr(v_obj, 'model_dump'):
        v_dict = v_obj.model_dump(by_alias=True)
    else:
        v_dict = v_obj

    objects = v_dict.get('objects', {})
    pages_list = objects.get('pages', [])

    # 3. Add or Update the entry for the new page ID
    page_entry_found = False
    for entry in pages_list:
        if entry.get('selector', {}).get('id') == target_page_id:
            # Ensure it is set to show
            entry['properties'] = {'showPage': {'expr': {'Literal': {'Value': 'true'}}}}
            page_entry_found = True
            break

    if not page_entry_found:
        # Add a new entry for P3 if it's missing
        new_entry = {
            "properties": {"showPage": {"expr": {"Literal": {"Value": "true"}}}},
            "selector": {"id": target_page_id}
        }
        pages_list.append(new_entry)

    # 4. Write back to the visual container
    objects['pages'] = pages_list
    v_dict['objects'] = objects

    # Re-assigning to the model (this depends on your Pydantic union structure)
    # If it was a dict, it stays a dict. If it was a model, we update attributes.
    if hasattr(v_obj, 'objects'):
        v_obj.objects = objects
    else:
        visual_container.root.visual = v_dict

# Execute update for P3
target_nav = next(v for v in target_page_wrapper.visuals if getattr(v.root.visual, 'visualType', '') == 'pageNavigator' or (isinstance(v.root.visual, dict) and v.root.visual.get('visualType') == 'pageNavigator'))
enable_page_in_navigator(target_nav, 'P3')

# Save the final definition
engine.save_report('/content/definition_output_v2')
print("Navigator updated for P3. Final report saved to /content/definition_output_v2")

Navigator updated for P3. Final report saved to /content/definition_output_v2


In [71]:
import json
import os

# Path to the visual in the new definition
# Note: Visual names might have changed if BPA was run, but we'll use the one identified previously (9b6b58405bb1540402d5)
nav_path = '/content/definition_output_v2/pages/P3/visuals/9b6b58405bb1540402d5/visual.json'

if os.path.exists(nav_path):
    with open(nav_path, 'r') as f:
        nav_data = json.load(f)

    print("--- Verified Navigator Configuration for P3 ---")
    pages_config = nav_data.get('visual', {}).get('objects', {}).get('pages', [])

    # Check if P3 exists and its status
    p3_entry = next((e for e in pages_config if e.get('selector', {}).get('id') == 'P3'), None)

    if p3_entry:
        print(f"Found P3 entry: {json.dumps(p3_entry, indent=2)}")
    else:
        print("P3 entry not found in the navigator configuration.")
else:
    print(f"Navigator visual file not found at: {nav_path}")

--- Verified Navigator Configuration for P3 ---
Found P3 entry: {
  "properties": {
    "showPage": {
      "expr": {
        "Literal": {
          "Value": "true"
        }
      }
    }
  },
  "selector": {
    "id": "P3"
  }
}


In [64]:
# Zipping the custom layout output
zip_result = zip(folder_path='/content/definition_output', zip_path='/content/definition_out')
print(zip_result)



/content/definition_out.zipcreated


# **PUSH CHANGES TO GIT**

In [ ]:
# push to git
from mach3_helpers import push_to_github
push_to_github(ROOT_PATH,commit_message='Save the models')

Push sequence complete.


In [72]:
import os

# List all .ipynb files in the repository recursively
notebook_files = []
for root, dirs, files in os.walk(ROOT_PATH):
    for file in files:
        if file.endswith('.ipynb'):
            # Create a path relative to the root for better readability
            rel_path = os.path.relpath(os.path.join(root, file), ROOT_PATH)
            notebook_files.append(rel_path)

print(f"Found {len(notebook_files)} notebook files in {REPO_NAME}:\n")
for nb in sorted(notebook_files):
    print(f"- {nb}")

Found 16 notebook files in Power_BI_Spark_Labs:

- Apache Spark/pyspark_basics_2025-08-21 12_10_32.ipynb
- Archive/2effff68-6ff2-42ef-8c3d-107cb479d762_83d04ac6-cb74-4a96-a06a-e0d5442aa126_python_vs_pyspark.ipynb
- Archive/Untitled1 (1).ipynb
- Archive/Untitled1 (2).ipynb
- Archive/Untitled1.ipynb
- Archive/Untitled2.ipynb
- Archive/Untitled3.ipynb
- Archive/Untitled4.ipynb
- Archive/Untitled5.ipynb
- Archive/Untitled6.ipynb
- Power BI/Automations/Microsoft Fabric/nb_model_refresh_schedule.ipynb
- Power BI/Automations/Microsoft Fabric/nb_populate_model_vw_bt_f.ipynb
- Power BI/Automations/Power Bi Desktop/Mach3/power_bi_objects_creation.ipynb
- Power BI/Automations/Power Bi Desktop/Mach3/report_development.ipynb
- Power BI/Automations/Power Bi Desktop/MachII/pbi_report_gen/base_articrafts/report_page_jsons/visuals/pageNavigator/Report_v01_generator.ipynb
- Power BI/Automations/Power Bi Desktop/REPORT_AUTOMATION/base_articrafts/report_page_jsons/visuals/pageNavigator/Report_v01_generato

In [73]:
import nbformat

# Define the target path based on the previous search results
target_nb_path = os.path.join(ROOT_PATH, 'Power BI/Automations/Power Bi Desktop/REPORT_AUTOMATION/base_articrafts/report_page_jsons/visuals/pageNavigator/Report_v01_generator.ipynb')

def read_notebook_cells(path):
    if not os.path.exists(path):
        return f"File not found: {path}"

    with open(path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    print(f"--- Content of {os.path.basename(path)} ---\n")
    for i, cell in enumerate(nb.cells):
        print(f"[Cell {i} - {cell.cell_type}]")
        # Print the first few lines of each cell to understand the logic
        lines = cell.source.split('\n')
        for line in lines[:10]:
            print(f"  {line}")
        if len(lines) > 10:
            print("  ...")
        print("-" * 30)

read_notebook_cells(target_nb_path)

--- Content of Report_v01_generator.ipynb ---

[Cell 0 - code]
  import json
  import os
  
  def read_json(file_path):
      try:
           with open(file_path,"r") as file:
              data = json.load(file)
           return data
      except Exception as e:
           print(f"error reading json {file_path}")
  ...
------------------------------
[Cell 1 - code]
  report_structure = {
                      "Supplier": [ "FillPage" , "POs" , "OpenPOs","InventoryBalance" , "ShelfAvailability" , 
                                  "Transactions" , "TransactionsData" , "Sales"] ,
                      "SupplierMetrics": [   "Data" , "IMF" ,"Return" ],
                      "SalesShare" : [ "Panel1"],
                      "FillRateDetails": ["Panel1"]
                     }
  
  metadata = generate_layout_v1(chapters_cnt= len(report_structure.keys()) )
  visual_coods = {v["id"]: v for v in metadata["visuals"]}
------------------------------
[Cell 2 - code]
  for key in report_structure

In [74]:
# Path to the second occurrence found in the MachII directory
target_nb_path_v2 = os.path.join(ROOT_PATH, 'Power BI/Automations/Power Bi Desktop/MachII/pbi_report_gen/base_articrafts/report_page_jsons/visuals/pageNavigator/Report_v01_generator.ipynb')

read_notebook_cells(target_nb_path_v2)

--- Content of Report_v01_generator.ipynb ---

[Cell 0 - code]
  import json
  import os
  
  def read_json(file_path):
      try:
           with open(file_path,"r") as file:
              data = json.load(file)
           return data
      except Exception as e:
           print(f"error reading json {file_path}")
  ...
------------------------------
[Cell 1 - code]
  report_structure = {
                      "Supplier": [ "FillPage" , "POs" , "OpenPOs","InventoryBalance" , "ShelfAvailability" , 
                                  "Transactions" , "TransactionsData" , "Sales"] ,
                      "SupplierMetrics": [   "Data" , "IMF" ,"Return" ],
                      "SalesShare" : [ "Panel1"],
                      "FillRateDetails": ["Panel1"]
                     }
  
  metadata = generate_layout_v1(chapters_cnt= len(report_structure.keys()) )
  visual_coods = {v["id"]: v for v in metadata["visuals"]}
------------------------------
[Cell 2 - code]
  for key in report_structure

Project Context: Power BI PBIR Automation
We are developing a programmatic flow to manage Power BI reports in the Fabric Enhanced Report Format (PBIR) using Python and Pydantic.

### **Latest Milestones & Progress:**
*   **BPA Engine & Schema Fixes:** Upgraded `FabricBPARules` to ensure strict schema compliance (e.g., `pages.json` structure) and implemented **Global Text Sync** to prevent Power BI Desktop crashes caused by ID mismatches.
*   **Top-Down Report Builder:** Integrated `FabricBuilderReport`, `FabricPage`, and `FabricVisual` classes to enable programmatic report assembly.
*   **Layout Engine:** Implemented an `ExecutiveLayout` class to dynamically calculate visual coordinates for headers and KPIs.
*   **Visual Support Expansion:** Added base classes for `FabricChart` and `FabricSlicer` to support diverse visual types in the builder flow.
*   **Navigator Synchronization:** Automated the logic to enable visibility for newly created pages within the `pageNavigator` visual.
*   **GitHub Synchronization:** Successfully pushed the enhanced builder logic, BPA rules, and updated Pydantic models to the `Power_BI_Spark_Labs` repository.

### **Next Steps:**
1. **Visual Detail Mapping:** Continue defining specific Pydantic property mappings for complex charts and slicers.
2. **Bookmark State Capture:** Implement logic to capture specific visual filters and visibility states into bookmarks programmatically.
3. **Template Library:** Begin defining standard layout templates (Operational, Financial, etc.) using the `LayoutEngine`.

*Updated: 2024-05-21*